# 02 - Chignolin CV Transformer

In [ ]:
# Requires: pip install -e .

import random
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from course_project.models.cv_core import CVCoreConfig, CVCoreInput, SharedCVCore
from course_project.utils import resolve_device

from course_project.peptide import (
    build_split_tensors,
    compute_norm_stats,
    ids_for_mutants,
    load_target_table,
)


### Loading mutant data

In [ ]:
device = torch.device(resolve_device('cuda'))
force_train = True
print('device:', device)

# Data 
peptide_dir = Path('../data/peptide')
packed_path = peptide_dir / 'hlda_trajectories_compact.pt'
tm_csv = peptide_dir / 'Tm.csv'
mfpt_csv = peptide_dir / 'mfpt_slice_thr0p34_tF0p25_tU0p57.csv'
evalue_csv = peptide_dir / 'hlda_evalues_thr0p34_tF0p25_tU0p57.csv'

num_runs = 1
train_mutant_count = 18
val_mutant_count = 8
# exclude_mutants = {'Y0R', 'D2R'}
exclude_mutants = {}

history = 1
frames_per_traj = 5000

# Model and training parms
hidden_size = 128
CVs = 7
token_sizes = (200, 60, CVs)
heads = 1
transformer_layers = 1
pre_pyramid_layers = 1
linear_cv_decoder = True
dropout = 0.005
batch_size = 256
learning_rate = 1e-4
weight_decay = 0.0
cls_weight = 0.5
time_lag_steps = 0
time_lag_weight = 0
epochs = 12
eval_every = 2

w_spearman = 0.2
w_pearson = 0.8

target_df = load_target_table(tm_csv, mfpt_csv, evalue_csv)
target_mutants = set(target_df['mutant'].astype(str).tolist())

packed = torch.load(packed_path, weights_only=False)
x_raw = packed['x'].float()
time_all = packed['time'].float()
offsets = packed['traj_offsets'].long()
labels = packed['labels'].long()
mutants = list(packed['mutants'])

x_all = x_raw
n_feat = int(x_all.shape[1])


uniq_mutants = sorted({
    m for m in set(mutants)
    if str(m).strip().upper() not in exclude_mutants and str(m) in target_mutants
})
print('packed path:', packed_path)
print('feature count:', n_feat)
print('all usable mutants:', len(uniq_mutants))


In [ ]:
import math


class SequenceHistoryAdapter(nn.Module):
    # Treat each descriptor like a node token: token i sees the history of descriptor i.
    def __init__(self, feat_dim, history, hidden_size):
        super().__init__()
        self.in_proj = nn.Linear(history, hidden_size)
        self.desc_emb = nn.Parameter(torch.randn(feat_dim, hidden_size) * 0.01)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.in_proj.weight)
        if self.in_proj.bias is not None:
            nn.init.zeros_(self.in_proj.bias)
        nn.init.normal_(self.desc_emb, mean=0.0, std=0.01)

    def build_core_input(self, x_hist) -> CVCoreInput:
        x_desc = x_hist.transpose(1, 2)
        tokens = self.in_proj(x_desc) + self.desc_emb.unsqueeze(0)
        mask = torch.ones(tokens.size(0), tokens.size(1), dtype=torch.bool, device=tokens.device)
        return CVCoreInput(tokens=tokens, mask=mask, local_skip=tokens)


class TriangularAttention(nn.Module):
    def __init__(self, hidden_size, heads, dropout):
        super().__init__()
        assert hidden_size % heads == 0
        self.hidden_size = hidden_size
        self.heads = heads
        self.head_dim = hidden_size // heads
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v1_proj = nn.Linear(hidden_size, hidden_size)
        self.v2_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.reset_parameters()

    def reset_parameters(self):
        for layer in (self.q_proj, self.k_proj, self.v1_proj, self.v2_proj, self.out_proj):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        b, n, _, d = x.shape
        q = self.q_proj(x).view(b, n, n, self.heads, self.head_dim)
        k = self.k_proj(x).view(b, n, n, self.heads, self.head_dim)
        v1 = self.v1_proj(x).view(b, n, n, self.heads, self.head_dim)
        v2 = self.v2_proj(x).view(b, n, n, self.heads, self.head_dim)

        scores = torch.einsum('bilhd,bljhd->biljh', q, k) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=2)
        attn = self.dropout(attn)
        values = v1.unsqueeze(3) * v2.unsqueeze(1)
        out = torch.einsum('biljh,biljhd->bijhd', attn, values).reshape(b, n, n, d)
        return self.out_proj(out)


class EdgeTransformerLayer(nn.Module):
    def __init__(self, hidden_size, heads, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = TriangularAttention(hidden_size, heads, dropout)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, 4 * hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * hidden_size, hidden_size),
        )
        self.dropout = nn.Dropout(dropout)
        self.reset_parameters()

    def reset_parameters(self):
        for module in self.ffn:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class TripleAttentionTokenLayer(nn.Module):
    def __init__(self, hidden_size, heads, dropout):
        super().__init__()
        self.left_proj = nn.Linear(hidden_size, hidden_size)
        self.right_proj = nn.Linear(hidden_size, hidden_size)
        self.edge_layer = EdgeTransformerLayer(hidden_size, heads, dropout)
        self.token_norm = nn.LayerNorm(hidden_size)
        self.token_mlp = nn.Sequential(
            nn.Linear(3 * hidden_size, 2 * hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * hidden_size, hidden_size),
        )
        self.dropout = nn.Dropout(dropout)
        self.reset_parameters()

    def reset_parameters(self):
        for layer in (self.left_proj, self.right_proj):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)
        for module in self.token_mlp:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def _init_edge_states(self, tokens):
        left = self.left_proj(tokens).unsqueeze(2)
        right = self.right_proj(tokens).unsqueeze(1)
        return left + right

    def forward(self, tokens):
        edge_states = self.edge_layer(self._init_edge_states(tokens))
        row_context = edge_states.mean(dim=2)
        col_context = edge_states.mean(dim=1)
        update = self.token_mlp(torch.cat([tokens, row_context, col_context], dim=-1))
        return tokens + self.dropout(update)


class ChignolinCVModel(nn.Module):
    def __init__(
        self,
        feat_dim,
        history,
        hidden,
        token_sizes,
        heads,
        token_layers,
        dropout,
        pre_pyramid_layers=0,
        linear_cv_decoder=True,
    ):
        super().__init__()
        self.feat_dim = int(feat_dim)
        self.cv_count = int(token_sizes[-1])
        self.linear_cv_decoder = bool(linear_cv_decoder)
        self.adapter = SequenceHistoryAdapter(feat_dim, history, hidden)
        self.pre_token_layers = nn.ModuleList([
            TripleAttentionTokenLayer(hidden, heads, dropout) for _ in range(pre_pyramid_layers)
        ])
        self.core = SharedCVCore(
            CVCoreConfig(
                hidden_size=hidden,
                output_dim=1,
                transformer_layers=token_layers,
                transformer_heads=heads,
                transformer_dropout=dropout,
                token_sizes=token_sizes,
                use_local_skip=True,
            )
        )
        if self.linear_cv_decoder:
            self.dv_head = nn.Linear(self.cv_count, self.feat_dim)
            self.tau_head = nn.Linear(self.cv_count, self.feat_dim)
        self.cls_head = nn.Sequential(
            nn.Linear(token_sizes[-1], hidden // 2),
            nn.GELU(),
            nn.Linear(hidden // 2, 1),
        )
        self.reset_parameters()

    def reset_parameters(self):
        for seq in (self.cls_head,):
            for module in seq.modules():
                if isinstance(module, nn.Linear):
                    nn.init.xavier_uniform_(module.weight)
                    if module.bias is not None:
                        nn.init.zeros_(module.bias)
        if self.linear_cv_decoder:
            for layer in (self.dv_head, self.tau_head):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)

    def forward(self, x_hist):
        core_input = self.adapter.build_core_input(x_hist)
        tokens = core_input.tokens
        for layer in self.pre_token_layers:
            tokens = layer(tokens)
        core_input = CVCoreInput(tokens=tokens, mask=core_input.mask, local_skip=tokens)
        self.core.encode(core_input)
        cv = self.core.last_cv
        if self.linear_cv_decoder:
            dv_pred = self.dv_head(cv)
            tau_pred = self.tau_head(cv)
        else:
            output = self.core(core_input)
            dv_pred = output.prediction.squeeze(-1)
            tau_pred = self.core.predict_tau(core_input).squeeze(-1)
            cv = output.cv
        cls_logit = self.cls_head(cv).squeeze(-1)
        return dv_pred, tau_pred, cls_logit, cv


### Training And Checkpoint Selection
Train the simulator backbone, save checkpoints, and track the best validation CV-selection score for each target metric.


In [ ]:
def run_epoch(loader, train_mode=True):
    model.train(train_mode)
    loss_sum = 0.0
    dv_loss_sum = 0.0
    tau_loss_sum = 0.0
    cls_loss_sum = 0.0
    n = 0
    y_true = []
    y_prob = []

    for xh, y_dv, y_tau, y_cls, traj_id in loader:
        xh = xh.to(device)
        y_dv = y_dv.to(device)
        y_tau = y_tau.to(device)
        y_cls = y_cls.to(device)
        traj_id = traj_id.to(device)

        with torch.set_grad_enabled(train_mode):
            dv_pred, tau_pred, cls_logit, cv = model(xh)
            loss_dv = F.mse_loss(dv_pred, y_dv)
            if time_lag_steps > 0 and time_lag_weight > 0:
                loss_tau = F.mse_loss(tau_pred, y_tau)
                loss = loss_dv + time_lag_weight * loss_tau
            else:
                loss_tau = torch.zeros((), device=loss_dv.device)
                loss = loss_dv
            if cls_weight > 0:
                loss_cls = F.binary_cross_entropy_with_logits(cls_logit, y_cls)
                loss = loss + cls_weight * loss_cls
            else:
                loss_cls = torch.zeros((), device=loss_dv.device)
            if train_mode:
                opt.zero_grad()
                loss.backward()
                opt.step()

        b = xh.size(0)
        n += b
        loss_sum += loss.item() * b
        dv_loss_sum += loss_dv.item() * b
        tau_loss_sum += loss_tau.item() * b
        cls_loss_sum += loss_cls.item() * b
        y_true.append(y_cls.detach().cpu())
        y_prob.append(torch.sigmoid(cls_logit).detach().cpu())

    y_true = torch.cat(y_true).numpy()
    y_prob = torch.cat(y_prob).numpy()
    y_pred = (y_prob >= 0.5).astype(np.int64)

    out = {
        'loss': loss_sum / n,
        'dv_mse': dv_loss_sum / n,
        'tau_mse': tau_loss_sum / n,
        'cls_bce': cls_loss_sum / n,
        'cls_acc': float((y_pred == y_true).mean()) if cls_weight > 0 else float('nan'),
    }
    return out


@torch.no_grad()
def cv_summary_df_from_split(x_tensor, sample_df):
    model.eval()
    cv_rows = []
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False, drop_last=False)
    for (xh,) in loader:
        xh = xh.to(device)
        _, _, _, cv = model(xh)
        cv_rows.append(cv.detach().cpu())
    cv_all = torch.cat(cv_rows, dim=0).numpy()
    cv_df = sample_df.reset_index(drop=True)
    base_cv_cols = []
    for i in range(cv_all.shape[1]):
        col = f'CV{i + 1}'
        cv_df[col] = cv_all[:, i]
        base_cv_cols.append(col)

    mean_df = cv_df.groupby('mutant', as_index=False)[base_cv_cols].mean()
    mean_df = mean_df.rename(columns={c: f'{c}_mean' for c in base_cv_cols})
    median_df = cv_df.groupby('mutant', as_index=False)[base_cv_cols].median()
    median_df = median_df.rename(columns={c: f'{c}_median' for c in base_cv_cols})
    return mean_df.merge(median_df, on='mutant', how='inner')

def merged_cv_target_for_split(x_tensor, sample_df):
    cv_df = cv_summary_df_from_split(x_tensor, sample_df)
    return cv_df.merge(target_df, on='mutant', how='inner')


def build_run_split_payload(run_seed):
    run_mutants = list(uniq_mutants)
    split_rng = np.random.default_rng(int(run_seed))
    split_rng.shuffle(run_mutants)
    train_mutants = list(run_mutants[:train_mutant_count])
    val_mutants = list(run_mutants[train_mutant_count:train_mutant_count + val_mutant_count])
    test_mutants = list(run_mutants[train_mutant_count + val_mutant_count:])

    train_ids = ids_for_mutants(mutants, train_mutants)
    val_ids = ids_for_mutants(mutants, val_mutants)
    test_ids = ids_for_mutants(mutants, test_mutants)

    x_mean, x_std, dv_mean, dv_std = compute_norm_stats(
        train_ids, history, time_lag_steps, frames_per_traj, x_all, time_all, offsets, n_feat
    )

    train_x, train_dv, train_dv_tau, train_cls, train_traj_id, train_samples_df = build_split_tensors(
        train_ids, history, time_lag_steps, frames_per_traj, x_all, time_all, offsets, labels, mutants, x_mean, x_std, dv_mean, dv_std
    )
    val_x, val_dv, val_dv_tau, val_cls, val_traj_id, val_samples_df = build_split_tensors(
        val_ids, history, time_lag_steps, frames_per_traj, x_all, time_all, offsets, labels, mutants, x_mean, x_std, dv_mean, dv_std
    )
    test_x, test_dv, test_dv_tau, test_cls, test_traj_id, test_samples_df = build_split_tensors(
        test_ids, history, time_lag_steps, frames_per_traj, x_all, time_all, offsets, labels, mutants, x_mean, x_std, dv_mean, dv_std
    )

    return {
        'train_mutants': train_mutants,
        'val_mutants': val_mutants,
        'test_mutants': test_mutants,
        'train_x': train_x,
        'train_dv': train_dv,
        'train_dv_tau': train_dv_tau,
        'train_cls': train_cls,
        'train_traj_id': train_traj_id,
        'train_samples_df': train_samples_df,
        'val_x': val_x,
        'val_dv': val_dv,
        'val_dv_tau': val_dv_tau,
        'val_cls': val_cls,
        'val_traj_id': val_traj_id,
        'val_samples_df': val_samples_df,
        'test_x': test_x,
        'test_dv': test_dv,
        'test_dv_tau': test_dv_tau,
        'test_cls': test_cls,
        'test_traj_id': test_traj_id,
        'test_samples_df': test_samples_df,
    }


@torch.no_grad()
def cv_frame_df_from_split(x_tensor, sample_df):
    model.eval()
    cv_rows = []
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False, drop_last=False)
    for (xh,) in loader:
        xh = xh.to(device)
        _, _, _, cv = model(xh)
        cv_rows.append(cv.detach().cpu())
    cv_all = torch.cat(cv_rows, dim=0).numpy()
    cv_df = sample_df.reset_index(drop=True)
    for i in range(cv_all.shape[1]):
        cv_df[f'CV{i + 1}'] = cv_all[:, i]
    return cv_df


def fit_r2(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    coeff = np.polyfit(x, y, deg=1)
    y_hat = coeff[0] * x + coeff[1]
    ss_res = float(np.sum((y - y_hat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return float(1.0 - ss_res / ss_tot)


def best_metric_from_merged(merged, metric):
    cv_cols = [c for c in merged.columns if c.startswith('CV') and (c.endswith('_mean') or c.endswith('_median'))]
    best = {'score': float('-inf'), 'r2': 0.0, 'cv': None, 'n': 0, 'spearman': 0.0, 'pearson': 0.0}
    for cv in cv_cols:
        sub = merged[[cv, metric]].dropna()
        n = int(len(sub))
        s = float(sub[cv].corr(sub[metric], method='spearman'))
        p = float(sub[cv].corr(sub[metric], method='pearson'))
        r2 = fit_r2(sub[cv].to_numpy(), sub[metric].to_numpy())
        score = w_spearman * abs(s) + w_pearson * abs(p)
        if score > best['score']:
            best = {'score': float(score), 'r2': float(r2), 'cv': cv, 'n': n, 'spearman': float(s), 'pearson': float(p)}
    return best


history_rows = []
val_metric_rows = []
selection_rows = []
selected_test_payload = {}
metrics = ['Tm', 'mfpt', 'log_mfpt_ratio', 'hlda_evalue']
best_by_metric = {
    metric: {
        'score': float('-inf'),
        'r2': 0.0,
        'epoch': None,
        'cv': None,
        'spearman': 0.0,
        'pearson': 0.0,
        'run_idx': None,
        'run_seed': None,
        'path': None,
    }
    for metric in metrics
}

run_name = 'chignolin_cv_transformer'
if pre_pyramid_layers > 0:
    run_name += f'_tri{pre_pyramid_layers}'
if linear_cv_decoder:
    run_name += '_lin'
run_dir = Path('../results') / run_name
run_dir.mkdir(parents=True, exist_ok=True)
print('run dir:', run_dir)
history_csv = run_dir / 'history.csv'
val_metric_csv = run_dir / 'val_metric_history.csv'
selection_csv = run_dir / 'selection_summary.csv'

need_train = force_train or not history_csv.exists() or not val_metric_csv.exists() or not selection_csv.exists()

if need_train:
    for run_idx in range(1, num_runs + 1):
        run_seed = int(torch.seed() % (2**32 - 1))
        random.seed(run_seed)
        np.random.seed(run_seed)
        torch.manual_seed(run_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(run_seed)

        split_payload = build_run_split_payload(run_seed)
        train_mutants = split_payload['train_mutants']
        val_mutants = split_payload['val_mutants']
        test_mutants = split_payload['test_mutants']
        train_x = split_payload['train_x']
        train_dv = split_payload['train_dv']
        train_dv_tau = split_payload['train_dv_tau']
        train_cls = split_payload['train_cls']
        train_traj_id = split_payload['train_traj_id']
        train_samples_df = split_payload['train_samples_df']
        val_x = split_payload['val_x']
        val_dv = split_payload['val_dv']
        val_dv_tau = split_payload['val_dv_tau']
        val_cls = split_payload['val_cls']
        val_traj_id = split_payload['val_traj_id']
        val_samples_df = split_payload['val_samples_df']
        test_x = split_payload['test_x']
        test_dv = split_payload['test_dv']
        test_dv_tau = split_payload['test_dv_tau']
        test_cls = split_payload['test_cls']
        test_traj_id = split_payload['test_traj_id']
        test_samples_df = split_payload['test_samples_df']

        train_loader = DataLoader(TensorDataset(train_x, train_dv, train_dv_tau, train_cls, train_traj_id), batch_size=batch_size, shuffle=True, drop_last=False)
        val_loader = DataLoader(TensorDataset(val_x, val_dv, val_dv_tau, val_cls, val_traj_id), batch_size=batch_size, shuffle=False, drop_last=False)
        test_loader = DataLoader(TensorDataset(test_x, test_dv, test_dv_tau, test_cls, test_traj_id), batch_size=batch_size, shuffle=False, drop_last=False)

        print('train mutants:', len(train_mutants), 'val mutants:', len(val_mutants), 'test mutants:', len(test_mutants))
        print('samples:', len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))

        model = ChignolinCVModel(
            feat_dim=n_feat,
            history=history,
            hidden=hidden_size,
            token_sizes=token_sizes,
            heads=heads,
            token_layers=transformer_layers,
            dropout=dropout,
            pre_pyramid_layers=pre_pyramid_layers,
            linear_cv_decoder=linear_cv_decoder,
        ).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

        run_ckpt_dir = run_dir / f'run_{run_idx:02d}'
        run_ckpt_dir.mkdir(parents=True, exist_ok=True)
        print('shared core token sizes:', model.core.token_sizes)
        print(f'time-lag steps={time_lag_steps} lambda={time_lag_weight}')

        for epoch in range(1, epochs + 1):
            tr = run_epoch(train_loader, train_mode=True)
            va = run_epoch(val_loader, train_mode=False)
            history_rows.append({
                'run_idx': run_idx,
                'run_seed': run_seed,
                'epoch': epoch,
                'train_loss': tr['loss'],
                'val_loss': va['loss'],
                'train_dv_mse': tr['dv_mse'],
                'val_dv_mse': va['dv_mse'],
                'train_tau_mse': tr['tau_mse'],
                'val_tau_mse': va['tau_mse'],
            })

            if epoch % eval_every != 0:
                print(
                    f"[run {run_idx:02d} ep {epoch:>3}/{epochs}] "
                    f"tr={tr['loss']:.3g} va={va['loss']:.3g} "
                    f"tau_mse={tr['tau_mse']:.3g}/{va['tau_mse']:.3g}"
                )
                continue

            merged_val = merged_cv_target_for_split(val_x, val_samples_df)
            ckpt_path = run_ckpt_dir / f'epoch_{epoch:04d}.pt'
            torch.save({'run_idx': run_idx, 'run_seed': run_seed, 'epoch': epoch, 'model_state_dict': model.state_dict()}, ckpt_path)

            best_text = 'none'
            best_score = float('-inf')
            for metric in metrics:
                best = best_metric_from_merged(merged_val, metric)
                val_metric_rows.append({
                    'run_idx': run_idx,
                    'run_seed': run_seed,
                    'epoch': epoch,
                    'metric': metric,
                    'val_best_score': float(best['score']),
                    'val_best_r2': float(best['r2']),
                    'val_best_cv': best['cv'],
                    'val_best_spearman': float(best['spearman']),
                    'val_best_pearson': float(best['pearson']),
                    'val_n': int(best['n']),
                })
                if best['score'] > best_by_metric[metric]['score']:
                    best_by_metric[metric] = {
                        'score': float(best['score']),
                        'r2': float(best['r2']),
                        'epoch': epoch,
                        'cv': best['cv'],
                        'spearman': float(best['spearman']),
                        'pearson': float(best['pearson']),
                        'run_idx': run_idx,
                        'run_seed': run_seed,
                        'path': str(ckpt_path),
                    }
                if best['score'] > best_score:
                    best_score = float(best['score'])
                    best_text = (
                        f"{metric}/{best['cv']} score={best['score']:.3g} "
                        f"R2={best['r2']:.3g} |S|={abs(best['spearman']):.3g} |P|={abs(best['pearson']):.3g}"
                    )

            print(
                f"[run {run_idx:02d} ep {epoch:>3}/{epochs}] "
                f"tr={tr['loss']:.3g} va={va['loss']:.3g} "
                f"tau_mse={tr['tau_mse']:.3g}/{va['tau_mse']:.3g} "
                f"best_cv={best_text}"
            )

    history_df = pd.DataFrame(history_rows)
    val_metric_history_df = pd.DataFrame(val_metric_rows)

    for metric in metrics:
        best = best_by_metric[metric]
        ckpt = torch.load(best['path'], map_location='cpu', weights_only=False)
        model = ChignolinCVModel(
            feat_dim=n_feat,
            history=history,
            hidden=hidden_size,
            token_sizes=token_sizes,
            heads=heads,
            token_layers=transformer_layers,
            dropout=dropout,
            pre_pyramid_layers=pre_pyramid_layers,
            linear_cv_decoder=linear_cv_decoder,
        ).to(device)
        model.load_state_dict(ckpt['model_state_dict'])

        split_payload = build_run_split_payload(best['run_seed'])
        val_merged = merged_cv_target_for_split(split_payload['val_x'], split_payload['val_samples_df'])
        test_merged = merged_cv_target_for_split(split_payload['test_x'], split_payload['test_samples_df'])
        cv_col = best['cv']
        val_sub = val_merged[[cv_col, metric, 'mutant']].dropna()
        test_sub = test_merged[[cv_col, metric, 'mutant']].dropna()
        val_r2 = fit_r2(val_sub[cv_col].to_numpy(), val_sub[metric].to_numpy())
        test_r2 = fit_r2(test_sub[cv_col].to_numpy(), test_sub[metric].to_numpy())
        val_s = float(val_sub[cv_col].corr(val_sub[metric], method='spearman'))
        val_p = float(val_sub[cv_col].corr(val_sub[metric], method='pearson'))
        test_s = float(test_sub[cv_col].corr(test_sub[metric], method='spearman'))
        test_p = float(test_sub[cv_col].corr(test_sub[metric], method='pearson'))

        selection_rows.append({
            'metric': metric,
            'best_run': int(best['run_idx']),
            'run_seed': int(best['run_seed']),
            'best_epoch': int(best['epoch']),
            'best_cv': cv_col,
            'val_score': float(best['score']),
            'val_r2': val_r2,
            'val_spearman': val_s,
            'val_pearson': val_p,
            'test_r2': test_r2,
            'test_spearman': test_s,
            'test_pearson': test_p,
            'test_n': int(len(test_sub)),
            'checkpoint_path': best['path'],
        })
        selected_test_payload[metric] = {
            'best_run': int(best['run_idx']),
            'run_seed': int(best['run_seed']),
            'best_epoch': int(best['epoch']),
            'cv': cv_col,
            'val_merged': val_merged,
            'test_merged': test_merged,
            'val_score': float(best['score']),
            'val_r2': val_r2,
            'val_spearman': val_s,
            'val_pearson': val_p,
            'test_r2': test_r2,
            'test_spearman': test_s,
            'test_pearson': test_p,
        }

    selection_summary_df = pd.DataFrame(selection_rows)
    history_df.to_csv(history_csv, index=False)
    val_metric_history_df.to_csv(val_metric_csv, index=False)
    selection_summary_df.to_csv(selection_csv, index=False)
    print(selection_summary_df)
else:
    print('using existing peptide run from', run_dir)
    history_df = pd.read_csv(history_csv)
    val_metric_history_df = pd.read_csv(val_metric_csv)
    selection_summary_df = pd.read_csv(selection_csv)
    for _, row in selection_summary_df.iterrows():
        metric = str(row['metric'])
        ckpt = torch.load(row['checkpoint_path'], map_location='cpu', weights_only=False)
        model = ChignolinCVModel(
            feat_dim=n_feat,
            history=history,
            hidden=hidden_size,
            token_sizes=token_sizes,
            heads=heads,
            token_layers=transformer_layers,
            dropout=dropout,
            pre_pyramid_layers=pre_pyramid_layers,
            linear_cv_decoder=linear_cv_decoder,
        ).to(device)
        model.load_state_dict(ckpt['model_state_dict'])
        split_payload = build_run_split_payload(int(row['run_seed']))
        val_merged = merged_cv_target_for_split(split_payload['val_x'], split_payload['val_samples_df'])
        test_merged = merged_cv_target_for_split(split_payload['test_x'], split_payload['test_samples_df'])
        selected_test_payload[metric] = {
            'best_run': int(row['best_run']),
            'run_seed': int(row['run_seed']),
            'best_epoch': int(row['best_epoch']),
            'cv': str(row['best_cv']),
            'val_merged': val_merged,
            'test_merged': test_merged,
            'val_score': float(row['val_score']),
            'val_r2': float(row['val_r2']),
            'val_spearman': float(row['val_spearman']),
            'val_pearson': float(row['val_pearson']),
            'test_r2': float(row['test_r2']),
            'test_spearman': float(row['test_spearman']),
            'test_pearson': float(row['test_pearson']),
        }
    first_row = selection_summary_df.iloc[0]
    ckpt = torch.load(first_row['checkpoint_path'], map_location='cpu', weights_only=False)
    model = ChignolinCVModel(
        feat_dim=n_feat,
        history=history,
        hidden=hidden_size,
        token_sizes=token_sizes,
        heads=heads,
        token_layers=transformer_layers,
        dropout=dropout,
        pre_pyramid_layers=pre_pyramid_layers,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])

if need_train and not selection_summary_df.empty:
    first_row = selection_summary_df.iloc[0]
    ckpt = torch.load(first_row['checkpoint_path'], map_location='cpu', weights_only=False)
    model = ChignolinCVModel(
        feat_dim=n_feat,
        history=history,
        hidden=hidden_size,
        token_sizes=token_sizes,
        heads=heads,
        token_layers=transformer_layers,
        dropout=dropout,
        pre_pyramid_layers=pre_pyramid_layers,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])


### Hyperparameter Tuning
Optional sweep over a small set of `02` training settings. The sweep reuses the same split and CV-selection logic and ranks configs by validation score and validation `R^2`.


In [ ]:
tune_force = False
tune_name = 'chignolin_cv_transformer_tuning'
tune_num_runs = 1
tune_metrics = ['Tm']

common_cfg = {
    'frames_per_traj': frames_per_traj,
    'train_mutant_count': train_mutant_count,
    'val_mutant_count': val_mutant_count,
    'learning_rate': learning_rate,
    'hidden_size': hidden_size,
    'CVs': CVs,
    'token_sizes': token_sizes,
    'transformer_layers': transformer_layers,
    'pre_pyramid_layers': pre_pyramid_layers,
    'dropout': dropout,
    'batch_size': batch_size,
    'cls_weight': cls_weight,
}

tune_grid = [
    {'name': 'base'},
    {'name': 'more_frames', 'frames_per_traj': 10000},
    {'name': 'train20_val8', 'train_mutant_count': 20, 'val_mutant_count': 8},
    {'name': 'train18_val10', 'train_mutant_count': 18, 'val_mutant_count': 10},
    {'name': 'lr3e4', 'learning_rate': 3e-4},
    {'name': 'lr3e5', 'learning_rate': 3e-5},
    {'name': 'hidden96', 'hidden_size': 96, 'token_sizes': (200, 60, CVs)},
    {'name': 'hidden160', 'hidden_size': 160, 'token_sizes': (240, 80, CVs)},
]


def train_and_score_tune_config(cfg, run_seed):
    global model, opt, batch_size, cls_weight, frames_per_traj, train_mutant_count, val_mutant_count
    prev_batch_size = batch_size
    prev_cls_weight = cls_weight
    prev_frames_per_traj = frames_per_traj
    prev_train_mutant_count = train_mutant_count
    prev_val_mutant_count = val_mutant_count

    batch_size = cfg['batch_size']
    cls_weight = cfg['cls_weight']
    frames_per_traj = cfg['frames_per_traj']
    train_mutant_count = cfg['train_mutant_count']
    val_mutant_count = cfg['val_mutant_count']

    split_payload = build_run_split_payload(run_seed)
    train_loader = DataLoader(
        TensorDataset(
            split_payload['train_x'], split_payload['train_dv'], split_payload['train_dv_tau'], split_payload['train_cls'], split_payload['train_traj_id']
        ),
        batch_size=cfg['batch_size'],
        shuffle=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        TensorDataset(
            split_payload['val_x'], split_payload['val_dv'], split_payload['val_dv_tau'], split_payload['val_cls'], split_payload['val_traj_id']
        ),
        batch_size=cfg['batch_size'],
        shuffle=False,
        drop_last=False,
    )

    model = ChignolinCVModel(
        feat_dim=n_feat,
        history=history,
        hidden=cfg['hidden_size'],
        token_sizes=cfg['token_sizes'],
        heads=heads,
        token_layers=cfg['transformer_layers'],
        dropout=cfg['dropout'],
        pre_pyramid_layers=cfg['pre_pyramid_layers'],
        linear_cv_decoder=linear_cv_decoder,
    ).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg['learning_rate'], weight_decay=weight_decay)

    history_rows_local = []
    best_by_metric_local = {
        metric: {'score': float('-inf'), 'r2': float('nan'), 'cv': None, 'epoch': None, 'spearman': float('nan'), 'pearson': float('nan')}
        for metric in tune_metrics
    }

    for epoch in range(1, epochs + 1):
        tr = run_epoch(train_loader, train_mode=True)
        va = run_epoch(val_loader, train_mode=False)
        row = {'epoch': epoch, 'train_loss': tr['loss'], 'val_loss': va['loss']}
        if epoch % eval_every == 0:
            merged_val = merged_cv_target_for_split(split_payload['val_x'], split_payload['val_samples_df'])
            for metric in tune_metrics:
                best = best_metric_from_merged(merged_val, metric)
                row[f'{metric}_best_score'] = float(best['score'])
                row[f'{metric}_best_r2'] = float(best['r2'])
                row[f'{metric}_best_cv'] = best['cv']
                if best['score'] > best_by_metric_local[metric]['score']:
                    best_by_metric_local[metric] = {
                        'score': float(best['score']),
                        'r2': float(best['r2']),
                        'cv': best['cv'],
                        'epoch': epoch,
                        'spearman': float(best['spearman']),
                        'pearson': float(best['pearson']),
                    }
        history_rows_local.append(row)

    batch_size = prev_batch_size
    cls_weight = prev_cls_weight
    frames_per_traj = prev_frames_per_traj
    train_mutant_count = prev_train_mutant_count
    val_mutant_count = prev_val_mutant_count

    return {
        'split_payload': split_payload,
        'history_df': pd.DataFrame(history_rows_local),
        'best_by_metric': best_by_metric_local,
    }


tune_dir = Path('../results') / tune_name
tune_dir.mkdir(parents=True, exist_ok=True)
tune_summary_csv = tune_dir / 'tune_summary.csv'

if tune_force or not tune_summary_csv.exists():
    tune_rows = []
    tune_history_frames = []
    for cfg_idx, cfg_spec in enumerate(tune_grid, start=1):
        cfg = dict(common_cfg)
        cfg.update(cfg_spec)
        cfg['CVs'] = int(cfg['token_sizes'][-1])
        print(f"[tune {cfg_idx}/{len(tune_grid)}] {cfg['name']}")
        print(json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}, indent=2))
        for run_idx in range(1, tune_num_runs + 1):
            run_seed = int(torch.seed() % (2**32 - 1))
            random.seed(run_seed)
            np.random.seed(run_seed)
            torch.manual_seed(run_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(run_seed)
            result = train_and_score_tune_config(cfg, run_seed)
            hist = result['history_df'].copy()
            hist['config_name'] = cfg['name']
            hist['run_idx'] = run_idx
            hist['run_seed'] = run_seed
            tune_history_frames.append(hist)
            row = {'config_name': cfg['name'], 'run_idx': run_idx, 'run_seed': run_seed, **{k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}}
            for metric in tune_metrics:
                best = result['best_by_metric'][metric]
                row[f'{metric}_val_best_score'] = float(best['score'])
                row[f'{metric}_val_best_r2'] = float(best['r2'])
                row[f'{metric}_val_best_cv'] = best['cv']
                row[f'{metric}_val_best_epoch'] = best['epoch']
                row[f'{metric}_val_best_spearman'] = float(best['spearman'])
                row[f'{metric}_val_best_pearson'] = float(best['pearson'])
            tune_rows.append(row)
    tune_summary_df = pd.DataFrame(tune_rows).sort_values([f'{tune_metrics[0]}_val_best_score', f'{tune_metrics[0]}_val_best_r2'], ascending=[False, False]).reset_index(drop=True)
    tune_history_df = pd.concat(tune_history_frames, ignore_index=True) if tune_history_frames else pd.DataFrame()
    tune_summary_df.to_csv(tune_summary_csv, index=False)
    if not tune_history_df.empty:
        tune_history_df.to_csv(tune_dir / 'tune_history.csv', index=False)
else:
    tune_summary_df = pd.read_csv(tune_summary_csv)
    tune_history_path = tune_dir / 'tune_history.csv'
    tune_history_df = pd.read_csv(tune_history_path) if tune_history_path.exists() else pd.DataFrame()

print('tuning summary:')
tune_summary_df


### Core Analysis
Review the selected checkpoints and inspect how the learned bottleneck CVs align with mutant-level metrics.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
for metric, color in [('Tm', 'tab:blue'), ('mfpt', 'tab:purple'), ('log_mfpt_ratio', 'tab:green'), ('hlda_evalue', 'tab:orange')]:
    sub = val_metric_history_df[val_metric_history_df['metric'] == metric]
    best_per_epoch = sub.groupby('epoch', as_index=False)[['val_best_score', 'val_best_r2']].max().sort_values('epoch')
    ax.plot(best_per_epoch['epoch'], best_per_epoch['val_best_score'], marker='o', ms=3, lw=1.2, color=color, label=f'{metric} score')
ax.set_xlabel('epoch')
ax.set_ylabel('best val score')
ax.set_title('Validation CV selection score across runs')
ax.grid(alpha=0.2)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
first_row = selection_summary_df.iloc[0]
split_payload = build_run_split_payload(int(first_row['run_seed']))
frame_cv_df = cv_frame_df_from_split(split_payload['val_x'], split_payload['val_samples_df'])
cv_cols = [c for c in frame_cv_df.columns if c.startswith('CV')]
assert len(cv_cols) >= 2
c1, c2 = cv_cols[0], cv_cols[1]

traj_label_df = split_payload['val_samples_df'][['traj_id']].drop_duplicates()
traj_label_df['label'] = [int(labels[int(tid)].item()) for tid in traj_label_df['traj_id']]
traj_to_mutant = split_payload['val_samples_df'][['traj_id', 'mutant']].drop_duplicates().reset_index(drop=True)
traj_label_df = traj_label_df.merge(traj_to_mutant, on='traj_id', how='left')

label0_traj = int(traj_label_df.loc[traj_label_df['label'] == 0, 'traj_id'].iloc[0])
label1_traj = int(traj_label_df.loc[traj_label_df['label'] == 1, 'traj_id'].iloc[0])

def normalize_pair(sub, xcol, ycol):
    x = sub[xcol].to_numpy(dtype=float)
    y = sub[ycol].to_numpy(dtype=float)
    x = (x - x.mean()) / (x.std() + 1e-8)
    y = (y - y.mean()) / (y.std() + 1e-8)
    return x, y

def plot_one_traj(traj_id, out_name, panel_title):
    sub = frame_cv_df[frame_cv_df['traj_id'] == int(traj_id)].reset_index(drop=True)
    row = traj_label_df[traj_label_df['traj_id'] == int(traj_id)].iloc[0]
    corr = np.abs(sub[cv_cols].corr(method='pearson').to_numpy(dtype=float))
    r = abs(float(sub[c1].corr(sub[c2], method='pearson')))
    x, y = normalize_pair(sub, c1, c2)

    fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2))
    im = axes[0].imshow(corr, vmin=0.0, vmax=1.0, cmap='viridis')
    axes[0].set_xticks(range(len(cv_cols)))
    axes[0].set_yticks(range(len(cv_cols)))
    axes[0].set_xticklabels(cv_cols, rotation=45, ha='right')
    axes[0].set_yticklabels(cv_cols)
    axes[0].set_title(f'{panel_title}: |CV-CV correlation|')
    for i in range(len(cv_cols)):
        for j in range(len(cv_cols)):
            axes[0].text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center', fontsize=9, color='white' if corr[i, j] > 0.55 else 'black')
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    axes[1].scatter(x, y, color='tab:blue', alpha=0.65, s=16)
    axes[1].set_title(f'{panel_title}: {c1} vs {c2}')
    axes[1].text(
        0.03, 0.97,
        f'traj {traj_id}\nmutant={row["mutant"]}\nlabel={int(row["label"])}\n|r|={r:.3f}',
        transform=axes[1].transAxes,
        va='top',
        ha='left',
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85, edgecolor='0.8'),
    )
    axes[1].set_xlabel(f'normalized {c1}')
    axes[1].set_ylabel(f'normalized {c2}')
    axes[1].grid(alpha=0.25)
    axes[1].set_box_aspect(1)

    fig.tight_layout()
    out_path = run_dir / out_name
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('saved:', out_path)

plot_one_traj(label0_traj, 'chignolin_val_single_label0_cv_view.png', 'Label 0 example')
plot_one_traj(label1_traj, 'chignolin_val_single_label1_cv_view.png', 'Label 1 example')


### Selected CV Correlations
Validation and test correlations for the validation-selected CV of each metric.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(21, 4.8))
for ax, metric in zip(np.asarray(axes).reshape(-1), ['Tm', 'mfpt', 'log_mfpt_ratio', 'hlda_evalue']):
    payload = selected_test_payload[metric]
    cv = payload['cv']
    sub = payload['val_merged'][[cv, metric, 'mutant']].dropna()
    ax.scatter(sub[cv], sub[metric], s=48, alpha=0.9)
    for _, r in sub.iterrows():
        ax.text(r[cv], r[metric], str(r['mutant']), fontsize=8, alpha=0.85)
    ax.set_title(
        f"VAL: {metric} vs {cv}\nrun {payload['best_run']}, ep {payload['best_epoch']}, score={payload['val_score']:.3f}, R²={payload['val_r2']:.3f}"
    )
    ax.set_xlabel(cv)
    ax.set_ylabel(metric)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(21, 4.8))
for ax, metric in zip(np.asarray(axes).reshape(-1), ['Tm', 'mfpt', 'log_mfpt_ratio', 'hlda_evalue']):
    payload = selected_test_payload[metric]
    cv = payload['cv']
    sub = payload['test_merged'][[cv, metric, 'mutant']].dropna()
    ax.scatter(sub[cv], sub[metric], s=48, alpha=0.9)
    for _, r in sub.iterrows():
        ax.text(r[cv], r[metric], str(r['mutant']), fontsize=8, alpha=0.85)
    ax.set_title(
        f"TEST: {metric} vs {cv}\nR²={payload['test_r2']:.3f}, |ρ|={abs(payload['test_spearman']):.3f}, |r|={abs(payload['test_pearson']):.3f}"
    )
    ax.set_xlabel(cv)
    ax.set_ylabel(metric)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


### Split-Robust Correlation Analysis
Re-evaluate CV selection under different validation split sizes and Pearson/Spearman weightings.


In [ ]:
analysis_metrics = ['Tm', 'mfpt', 'log_mfpt_ratio', 'hlda_evalue']
analysis_summary_types = ('mean', 'median')
analysis_weight_options = (
    (0.8, 0.2),
    (0.5, 0.5),
    (0.2, 0.8),
)
analysis_val_mutant_counts = (4, 6, 8)
selected_weight_pearson = 0.5
selected_weight_spearman = 0.5
selected_weight_label = f'P{selected_weight_pearson:.1f}/S{selected_weight_spearman:.1f}'
selected_analysis_val_mutants = 6
analysis_success_threshold = 0.5
metric_label_map = {
    'Tm': 'Tm',
    'mfpt': 'MFPT',
    'log_mfpt_ratio': 'log MFPT ratio',
    'hlda_evalue': 'HLDA eig',
}
print('analysis summary types:', analysis_summary_types)
print('weight options:', analysis_weight_options)
print('analysis val mutant counts:', analysis_val_mutant_counts)
print('selected weight label:', selected_weight_label)
print('selected analysis val mutants:', selected_analysis_val_mutants)


In [ ]:
from scipy import stats as scipy_stats

def analysis_score(pearson_value, spearman_value, w_pearson, w_spearman):
    return w_pearson * abs(float(pearson_value)) + w_spearman * abs(float(spearman_value))


def metric_stats(df, cv_col, metric):
    sub = df[[cv_col, metric, 'mutant']].dropna()
    x = sub[cv_col].to_numpy(dtype=float)
    y = sub[metric].to_numpy(dtype=float)
    try:
        pearson = scipy_stats.pearsonr(x, y)
        pearson_r = float(pearson.statistic)
        pearson_p = float(pearson.pvalue)
    except Exception:
        pearson_r = float('nan')
        pearson_p = float('nan')
    try:
        spearman = scipy_stats.spearmanr(x, y)
        spearman_r = float(spearman.statistic)
        spearman_p = float(spearman.pvalue)
    except Exception:
        spearman_r = float('nan')
        spearman_p = float('nan')
    return {
        'n': int(len(sub)),
        'r2': fit_r2(x, y),
        'spearman': spearman_r,
        'spearman_p': spearman_p,
        'pearson': pearson_r,
        'pearson_p': pearson_p,
        'points': sub,
    }


def candidate_cv_cols(df):
    cols = []
    for col in df.columns:
        if col.startswith('CV') and any(col.endswith(f'_{kind}') for kind in analysis_summary_types):
            cols.append(col)
    return cols


def compact_p(p_value):
    if np.isnan(p_value):
        return 'nan'
    if p_value < 1e-3:
        return '<1e-3'
    return f'{p_value:.3f}'


def signif_stars(p_value):
    if np.isnan(p_value):
        return 'na'
    if p_value < 0.001:
        return '***'
    if p_value < 0.01:
        return '**'
    if p_value < 0.05:
        return '*'
    return 'ns'


def corr_p_from_r(r_value, n_points):
    if np.isnan(r_value) or n_points is None or n_points < 3:
        return float('nan')
    if abs(r_value) >= 1.0:
        return 0.0
    denom = max(1e-12, 1.0 - float(r_value) ** 2)
    t_stat = abs(float(r_value)) * np.sqrt((n_points - 2) / denom)
    return float(2.0 * scipy_stats.t.sf(t_stat, df=n_points - 2))


def bootstrap_corr_ci(x, y, method='pearson', n_boot=2000, seed=0):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 4:
        return (float('nan'), float('nan'))
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        xb = x[idx]
        yb = y[idx]
        try:
            if method == 'pearson':
                value = float(np.corrcoef(xb, yb)[0, 1])
            else:
                value = float(scipy_stats.spearmanr(xb, yb).statistic)
        except Exception:
            value = float('nan')
        if np.isfinite(value):
            values.append(value)
    if len(values) < max(50, n_boot // 10):
        return (float('nan'), float('nan'))
    return tuple(np.quantile(values, [0.025, 0.975]))


def leave_one_out_min_abs_corr(x, y, method='pearson'):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 4:
        return float('nan')
    values = []
    for drop_idx in range(n):
        mask = np.ones(n, dtype=bool)
        mask[drop_idx] = False
        try:
            if method == 'pearson':
                value = float(np.corrcoef(x[mask], y[mask])[0, 1])
            else:
                value = float(scipy_stats.spearmanr(x[mask], y[mask]).statistic)
        except Exception:
            value = float('nan')
        values.append(abs(value) if np.isfinite(value) else float('nan'))
    finite_values = [value for value in values if np.isfinite(value)]
    return min(finite_values) if finite_values else float('nan')


def split_payload_from_mutants(train_mutants, val_mutants, test_mutants):
    train_ids = ids_for_mutants(mutants, train_mutants)
    val_ids = ids_for_mutants(mutants, val_mutants)
    test_ids = ids_for_mutants(mutants, test_mutants)
    x_mean, x_std, dv_mean, dv_std = compute_norm_stats(
        train_ids, history, time_lag_steps, frames_per_traj, x_all, time_all, offsets, n_feat
    )
    train_x, train_dv, train_dv_tau, train_cls, train_traj_id, train_samples_df = build_split_tensors(
        train_ids, history, time_lag_steps, frames_per_traj, x_all, time_all, offsets, labels, mutants, x_mean, x_std, dv_mean, dv_std
    )
    val_x, val_dv, val_dv_tau, val_cls, val_traj_id, val_samples_df = build_split_tensors(
        val_ids, history, time_lag_steps, frames_per_traj, x_all, time_all, offsets, labels, mutants, x_mean, x_std, dv_mean, dv_std
    )
    test_x, test_dv, test_dv_tau, test_cls, test_traj_id, test_samples_df = build_split_tensors(
        test_ids, history, time_lag_steps, frames_per_traj, x_all, time_all, offsets, labels, mutants, x_mean, x_std, dv_mean, dv_std
    )
    return {
        'train_mutants': list(train_mutants),
        'val_mutants': list(val_mutants),
        'test_mutants': list(test_mutants),
        'train_x': train_x,
        'train_samples_df': train_samples_df,
        'val_x': val_x,
        'val_samples_df': val_samples_df,
        'test_x': test_x,
        'test_samples_df': test_samples_df,
    }


def load_model_for_checkpoint(checkpoint_path):
    model_local = ChignolinCVModel(
        feat_dim=n_feat,
        history=history,
        hidden=hidden_size,
        token_sizes=token_sizes,
        heads=heads,
        token_layers=transformer_layers,
        dropout=dropout,
        pre_pyramid_layers=pre_pyramid_layers,
        linear_cv_decoder=linear_cv_decoder,
    ).to(device)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model_local.load_state_dict(ckpt['model_state_dict'])
    model_local.eval()
    return model_local


@torch.no_grad()
def cv_summary_df_for_model(model_local, x_tensor, sample_df):
    rows = []
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False, drop_last=False)
    for (xh,) in loader:
        xh = xh.to(device)
        _, _, _, cv = model_local(xh)
        rows.append(cv.detach().cpu())
    cv_all = torch.cat(rows, dim=0).numpy()
    cv_df = sample_df.reset_index(drop=True).copy()
    base_cols = []
    for i in range(cv_all.shape[1]):
        col = f'CV{i + 1}'
        cv_df[col] = cv_all[:, i]
        base_cols.append(col)
    mean_df = cv_df.groupby('mutant', as_index=False)[base_cols].mean()
    mean_df = mean_df.rename(columns={c: f'{c}_mean' for c in base_cols})
    median_df = cv_df.groupby('mutant', as_index=False)[base_cols].median()
    median_df = median_df.rename(columns={c: f'{c}_median' for c in base_cols})
    return mean_df.merge(median_df, on='mutant', how='inner')


analysis_rows = []
analysis_cache = {}
for _, row in selection_summary_df[['best_run', 'run_seed']].drop_duplicates().sort_values('best_run').iterrows():
    run_idx = int(row['best_run'])
    run_seed = int(row['run_seed'])
    run_mutants = list(uniq_mutants)
    split_rng = np.random.default_rng(run_seed)
    split_rng.shuffle(run_mutants)
    train_mutants = list(run_mutants[:train_mutant_count])
    holdout_mutants = list(run_mutants[train_mutant_count:])
    ckpt_dir = run_dir / f'run_{run_idx:02d}'
    for analysis_val_mutants in analysis_val_mutant_counts:
        if analysis_val_mutants >= len(holdout_mutants):
            continue
        val_mutants = holdout_mutants[:analysis_val_mutants]
        test_mutants = holdout_mutants[analysis_val_mutants:]
        split_payload = split_payload_from_mutants(train_mutants, val_mutants, test_mutants)
        for checkpoint_path in sorted(ckpt_dir.glob('epoch_*.pt')):
            epoch = int(checkpoint_path.stem.split('_')[-1])
            model_local = load_model_for_checkpoint(str(checkpoint_path))
            val_merged = cv_summary_df_for_model(model_local, split_payload['val_x'], split_payload['val_samples_df']).merge(target_df, on='mutant', how='inner')
            test_merged = cv_summary_df_for_model(model_local, split_payload['test_x'], split_payload['test_samples_df']).merge(target_df, on='mutant', how='inner')
            analysis_cache[(run_idx, analysis_val_mutants, epoch)] = {
                'checkpoint_path': str(checkpoint_path),
                'run_seed': run_seed,
                'val_mutants': list(val_mutants),
                'test_mutants': list(test_mutants),
                'val_merged': val_merged,
                'test_merged': test_merged,
            }
            cv_cols = candidate_cv_cols(val_merged)
            for metric in analysis_metrics:
                for cv_col in cv_cols:
                    val_stats = metric_stats(val_merged, cv_col, metric)
                    test_stats = metric_stats(test_merged, cv_col, metric)
                    analysis_rows.append({
                        'run_idx': run_idx,
                        'run_seed': run_seed,
                        'analysis_val_mutants': analysis_val_mutants,
                        'analysis_test_mutants': len(test_mutants),
                        'epoch': epoch,
                        'metric': metric,
                        'cv': cv_col,
                        'summary_type': cv_col.rsplit('_', 1)[-1],
                        'val_r2': val_stats['r2'],
                        'val_spearman': val_stats['spearman'],
                        'val_pearson': val_stats['pearson'],
                        'test_r2': test_stats['r2'],
                        'test_spearman': test_stats['spearman'],
                        'test_pearson': test_stats['pearson'],
                    })

analysis_candidates_df = pd.DataFrame(analysis_rows).sort_values(['analysis_val_mutants', 'run_idx', 'epoch', 'metric', 'cv']).reset_index(drop=True)
print('analysis candidates:', analysis_candidates_df.shape)


In [ ]:
def build_weighted_run_report(weight_pearson, weight_spearman):
    weighted_df = analysis_candidates_df.copy()
    weighted_df['val_score'] = (
        weight_pearson * weighted_df['val_pearson'].abs()
        + weight_spearman * weighted_df['val_spearman'].abs()
    )
    run_selection_df = (
        weighted_df
        .sort_values(
            ['analysis_val_mutants', 'run_idx', 'metric', 'val_score', 'test_pearson', 'test_spearman'],
            ascending=[True, True, True, False, False, False],
        )
        .groupby(['analysis_val_mutants', 'run_idx', 'metric'], as_index=False)
        .first()
    )
    run_selection_df['val_abs_pearson'] = run_selection_df['val_pearson'].abs()
    run_selection_df['val_abs_spearman'] = run_selection_df['val_spearman'].abs()
    run_selection_df['test_abs_pearson'] = run_selection_df['test_pearson'].abs()
    run_selection_df['test_abs_spearman'] = run_selection_df['test_spearman'].abs()
    run_selection_df['test_score'] = (
        weight_pearson * run_selection_df['test_abs_pearson']
        + weight_spearman * run_selection_df['test_abs_spearman']
    )
    run_selection_df['generalization_gap'] = run_selection_df['val_score'] - run_selection_df['test_score']
    run_selection_df['weight_label'] = f'P{weight_pearson:.1f}/S{weight_spearman:.1f}'
    return run_selection_df


def ensure_weighted_run_reports():
    global weighted_run_reports
    if 'weighted_run_reports' not in globals():
        weighted_run_reports = {
            f'P{wp:.1f}/S{ws:.1f}': build_weighted_run_report(wp, ws)
            for wp, ws in analysis_weight_options
        }
    return weighted_run_reports


def get_selected_report_df(weight_label):
    return ensure_weighted_run_reports()[weight_label]


def get_selected_row(report_df, analysis_val_mutants, run_idx, metric):
    return report_df[
        (report_df['analysis_val_mutants'] == analysis_val_mutants)
        & (report_df['run_idx'] == run_idx)
        & (report_df['metric'] == metric)
    ].iloc[0]


def evaluate_selected_row_support(row, seed_offset=0):
    payload = analysis_cache[(int(row['run_idx']), int(row['analysis_val_mutants']), int(row['epoch']))]
    metric = str(row['metric'])
    cv = str(row['cv'])
    test_points = payload['test_merged'][[cv, metric, 'mutant']].dropna()
    x = test_points[cv].to_numpy(dtype=float)
    y = test_points[metric].to_numpy(dtype=float)
    n_points = int(len(test_points))
    pearson_r = float(row['test_pearson'])
    spearman_r = float(row['test_spearman'])
    pearson_p = corr_p_from_r(pearson_r, n_points)
    spearman_p = corr_p_from_r(spearman_r, n_points)
    pearson_ci_low, pearson_ci_high = bootstrap_corr_ci(x, y, method='pearson', seed=seed_offset)
    spearman_ci_low, spearman_ci_high = bootstrap_corr_ci(x, y, method='spearman', seed=seed_offset + 1)
    loo_min_abs_pearson = leave_one_out_min_abs_corr(x, y, method='pearson')
    loo_min_abs_spearman = leave_one_out_min_abs_corr(x, y, method='spearman')
    return {
        'test_n': n_points,
        'pearson_p': pearson_p,
        'spearman_p': spearman_p,
        'pearson_sig': signif_stars(pearson_p),
        'spearman_sig': signif_stars(spearman_p),
        'pearson_ci_low': pearson_ci_low,
        'pearson_ci_high': pearson_ci_high,
        'spearman_ci_low': spearman_ci_low,
        'spearman_ci_high': spearman_ci_high,
        'loo_min_abs_pearson': loo_min_abs_pearson,
        'loo_min_abs_spearman': loo_min_abs_spearman,
    }


def build_support_summary(report_df, analysis_val_mutants):
    selected = report_df[report_df['analysis_val_mutants'] == analysis_val_mutants].copy()
    rows = []
    for row_idx, (_, row) in enumerate(selected.sort_values(['run_idx', 'metric']).iterrows()):
        support = evaluate_selected_row_support(row, seed_offset=2000 + row_idx)
        rows.append({
            'run_idx': int(row['run_idx']),
            'metric': str(row['metric']),
            'cv': str(row['cv']),
            'summary_type': str(row['summary_type']),
            'test_n': support['test_n'],
            'test_pearson': float(row['test_pearson']),
            'pearson_p': support['pearson_p'],
            'pearson_sig': support['pearson_sig'],
            'pearson_ci': f"[{support['pearson_ci_low']:.2f}, {support['pearson_ci_high']:.2f}]" if np.isfinite(support['pearson_ci_low']) and np.isfinite(support['pearson_ci_high']) else 'nan',
            'pearson_loo_min_abs': support['loo_min_abs_pearson'],
            'test_spearman': float(row['test_spearman']),
            'spearman_p': support['spearman_p'],
            'spearman_sig': support['spearman_sig'],
            'spearman_ci': f"[{support['spearman_ci_low']:.2f}, {support['spearman_ci_high']:.2f}]" if np.isfinite(support['spearman_ci_low']) and np.isfinite(support['spearman_ci_high']) else 'nan',
            'spearman_loo_min_abs': support['loo_min_abs_spearman'],
        })
    return pd.DataFrame(rows)


def build_score_tables(report_df, analysis_val_mutants, metrics=None):
    metrics = analysis_metrics if metrics is None else metrics
    sub = report_df[report_df['analysis_val_mutants'] == analysis_val_mutants].copy()
    score_mat = (
        sub.pivot(index='run_idx', columns='metric', values='test_score')
        .reindex(index=sorted(sub['run_idx'].unique()), columns=metrics)
    )
    pearson_mat = (
        sub.pivot(index='run_idx', columns='metric', values='test_abs_pearson')
        .reindex(index=score_mat.index, columns=metrics)
    )
    spearman_mat = (
        sub.pivot(index='run_idx', columns='metric', values='test_abs_spearman')
        .reindex(index=score_mat.index, columns=metrics)
    )
    hit_rate = (score_mat >= analysis_success_threshold).mean(axis=0)
    test_mutant_count = int(sub['analysis_test_mutants'].iloc[0])
    return sub, score_mat, pearson_mat, spearman_mat, hit_rate, test_mutant_count


def render_score_matrix(report_df, weight_label, analysis_val_mutants, title_prefix='Test score'):
    metrics = analysis_metrics
    metric_labels = [metric_label_map.get(metric, metric) for metric in metrics]
    sub, score_mat, pearson_mat, spearman_mat, hit_rate, test_mutant_count = build_score_tables(report_df, analysis_val_mutants, metrics=metrics)
    print('Metric labels:', ', '.join(f'{short}={full}' for short, full in zip(metric_labels, metrics)))
    print(f'{weight_label}, val mutants = {analysis_val_mutants}, test mutants = {test_mutant_count}')
    fig, axes = plt.subplots(1, 2, figsize=(17.8, 5.6), gridspec_kw={'width_ratios': [6.6, 1.8]})
    im = axes[0].imshow(score_mat.to_numpy(dtype=float), vmin=0.0, vmax=1.0, cmap='viridis', aspect='auto')
    axes[0].set_xticks(range(len(metrics)))
    axes[0].set_xticklabels(metric_labels, rotation=0, ha='center', fontsize=11)
    axes[0].set_yticks(range(len(score_mat.index)))
    axes[0].set_yticklabels([f'run {int(v)}' for v in score_mat.index])
    axes[0].set_title(f'{title_prefix} {weight_label}, val mutants = {analysis_val_mutants}, test mutants = {test_mutant_count}')
    for i in range(score_mat.shape[0]):
        for j in range(score_mat.shape[1]):
            score = float(score_mat.iloc[i, j])
            rp = float(pearson_mat.iloc[i, j])
            rs = float(spearman_mat.iloc[i, j])
            row = get_selected_row(sub, analysis_val_mutants, score_mat.index[i], metrics[j])
            payload = analysis_cache[(int(row['run_idx']), int(row['analysis_val_mutants']), int(row['epoch']))]
            test_points = payload['test_merged'][[str(row['cv']), metrics[j], 'mutant']].dropna()
            n_points = int(len(test_points))
            rp_p = corr_p_from_r(rp, n_points)
            rs_p = corr_p_from_r(rs, n_points)
            axes[0].text(
                j,
                i,
                f'{score:.2f} r={rp:.2f} p={compact_p(rp_p)} ρ={rs:.2f} p={compact_p(rs_p)}',
                ha='center',
                va='center',
                fontsize=9,
                color='white' if score > 0.55 else 'black',
            )
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
    axes[1].bar(range(len(metrics)), hit_rate.to_numpy(dtype=float), color='tab:blue', alpha=0.85, width=0.62)
    axes[1].axhline(1.0, color='black', linewidth=0.8)
    axes[1].set_xticks(range(len(metrics)))
    axes[1].set_xticklabels(metric_labels, rotation=0, ha='center', fontsize=11)
    axes[1].set_ylim(0.0, 1.05)
    axes[1].set_ylabel('fraction of runs above threshold')
    axes[1].set_title(f'Hit rate, test mutants = {test_mutant_count} score ≥ {analysis_success_threshold:.2f}')
    axes[1].grid(alpha=0.2, axis='y')
    for i, value in enumerate(hit_rate.to_numpy(dtype=float)):
        axes[1].text(i, value + 0.03, f'{value:.2f}', ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.show()


weighted_run_reports = ensure_weighted_run_reports()


In [ ]:
summary_cols = [
    'analysis_val_mutants', 'analysis_test_mutants', 'run_idx', 'metric', 'epoch', 'cv', 'summary_type',
    'val_score', 'val_r2', 'val_spearman', 'val_pearson',
    'test_r2', 'test_spearman', 'test_pearson'
]
selected_report_df = get_selected_report_df(selected_weight_label)
print('Pinned per-run selected checkpoints')
selected_report_df[
    selected_report_df['analysis_val_mutants'] == selected_analysis_val_mutants
][summary_cols].sort_values(['run_idx', 'metric']).round(3)


In [ ]:
selected_report_df = get_selected_report_df(selected_weight_label)
support_summary_df = build_support_summary(selected_report_df, selected_analysis_val_mutants)
print('Pinned support summary from selected test points')
support_summary_df.sort_values(['metric', 'run_idx']).round(3)


In [ ]:
selected_report_df = get_selected_report_df(selected_weight_label)
render_score_matrix(
    selected_report_df,
    selected_weight_label,
    selected_analysis_val_mutants,
    title_prefix='Pinned test score',
)


### All Weight Options
Compare the split-robust reports across all Pearson/Spearman weightings and validation split sizes.


In [ ]:
weighted_run_reports = ensure_weighted_run_reports()
for weight_label, report_df in weighted_run_reports.items():
    for selected_analysis_val_mutants in analysis_val_mutant_counts:
        render_score_matrix(report_df, weight_label, selected_analysis_val_mutants)


In [ ]:
weighted_run_reports = ensure_weighted_run_reports()
weighted_strength_rows = []
for weight_label, report_df in weighted_run_reports.items():
    tmp = (
        report_df
        .groupby(['analysis_val_mutants', 'metric'], as_index=False)[['test_score', 'test_abs_pearson', 'test_abs_spearman']]
        .mean()
    )
    tmp['weight_label'] = weight_label
    weighted_strength_rows.append(tmp)
metric_strength_df = pd.concat(weighted_strength_rows, ignore_index=True)
print('Average test strength by metric, analysis split choice, and weight setting')
metric_strength_df.sort_values(['weight_label', 'analysis_val_mutants', 'test_score'], ascending=[True, True, False]).round(3)
